In [ ]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.datasets import make_classification 
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.feature_selection import SelectKBest, SelectFpr, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import SnowballStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
import re
import string
from bs4 import BeautifulSoup
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix

In [ ]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [ ]:
df.drop_duplicates(inplace=True)
df = df.dropna()
test_ids = eva['Id'].copy()
eva['article'].fillna("", inplace=True)

In [ ]:
import re
import string
from bs4 import BeautifulSoup

def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

from bs4 import BeautifulSoup, Comment
import re

def clean_html_professional(html_content):
    if not isinstance(html_content, str):
        return ""

    soup = BeautifulSoup(html_content, "html.parser") # lxml è più veloce di html.parser

    # 1. SALVARE I METADATA PREZIOSI (Opzionale, ma consigliato)
    meta_desc = ""
    meta_tag = soup.find('meta', attrs={'name': 'description'})
    if meta_tag and 'content' in meta_tag.attrs:
        meta_desc = meta_tag['content']

    # 2. GESTIONE JSON-LD (Dati strutturati)
    # Se vuoi estrarre info specifiche da qui, fallo ora prima di rimuovere gli script.

    # 3. RIMOZIONE DEL RUMORE (Decompose distrugge il tag e il suo contenuto)
    for tag in soup(['script', 'style', 'noscript', 'iframe', 'svg', 'header', 'footer', 'nav']):
        # Eccezione: se vuoi tenere JSON-LD, controlla l'attributo type prima di decomporre script
        tag.decompose()

    # Rimuovi commenti HTML
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for c in comments:
        c.extract()

    # 4. ESTRAZIONE ATTRIBUTI NASCOSTI (Come chiedevi per le immagini)
    # Sostituiamo l'immagine con il suo testo alternativo per non perderlo
    for img in soup.find_all('img'):
        if 'alt' in img.attrs and img['alt']:
            # Sostituisce <img> con il testo dell'alt
            img.replace_with(f" {img['alt']} ") 
    
    # 5. ESTRAZIONE TESTO (Gestione spaziature)
    # 'separator=" "' è CRUCIALE. Altrimenti <p>Hello</p><p>World</p> diventa "HelloWorld"
    text = soup.get_text(separator=' ', strip=True)
    
    # Aggiungiamo la meta description recuperata all'inizio
    final_text = f"{text} {meta_desc}"

    # 6. PULIZIA FINALE ENTITIES E SPAZI
    # BeautifulSoup gestisce le entità (&amp;) automaticamente in get_text(), 
    # ma rimuoviamo spazi multipli
    final_text = re.sub(r'\s+', ' ', final_text).strip()

    return final_text

for i, article in zip(df.index, df['article']):
    df.loc[i, 'article'] = clean_html_professional(article)

for i, article in zip(eva.index, eva['article']):
    eva.loc[i, 'article'] = clean_html_professional(article)


In [ ]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
df['text'] = df['source'] + ' ' + df['source'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['article'] + ' ' + df['article']
eva['text'] = eva['source'] + ' ' + eva['source'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['article'] + ' ' + eva['article']

df.columns

In [ ]:
weekdays = []
daytime = []
for day in df['timestamp']:
    if day == '0000-00-00 00:00:00':
        week_day = -1
        hour_day = -1
    else:
        week_day = pd.Timestamp(day).day_of_week
        hour = pd.Timestamp(day).hour
        if hour > 5 and hour <= 14:
            hour_day = 1        #morning
        elif hour > 14 and hour <= 21:
            hour_day = 2        #afternoon
        elif (hour > 21 and hour <= 23) or (hour >= 0 and hour <= 5):
            hour_day = 3        #night

    daytime.append(hour_day)
    weekdays.append(week_day)

df['day_of_week'] = weekdays
df['moment_of_day'] = daytime


weekdays_eva = []
daytime_eva = []
for day in eva['timestamp']:
    if day == '0000-00-00 00:00:00':
        week_day = -1
        hour_day = -1
    else:
        week_day = pd.Timestamp(day).day_of_week
        hour = pd.Timestamp(day).hour
        if hour > 5 and hour <= 14:
            hour_day = 1        #morning
        elif hour > 14 and hour <= 21:
            hour_day = 2        #afternoon
        elif (hour > 21 and hour <= 23) or (hour >= 0 and hour <= 5):
            hour_day = 3        #night

    daytime_eva.append(hour_day)
    weekdays_eva.append(week_day)

eva['day_of_week'] = weekdays_eva
eva['moment_of_day'] = daytime_eva

In [ ]:
y = df['label']
df.drop('label', inplace=True, axis=1)
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)


In [ ]:
encoder = OneHotEncoder(min_frequency=70, handle_unknown='ignore')

vectorizer_svd = Pipeline([( 'vect', TfidfVectorizer(
        max_features=50000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)),
        ('svd', TruncatedSVD(
            n_components=1000, 
            algorithm='randomized'))])

vectorizer = TfidfVectorizer(
        max_features=60000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)

vectorizer2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=5,
    max_features=30000
)

preprocessor = ColumnTransformer(transformers=[
    ('source', encoder, ['source', 'day_of_week', 'moment_of_day']),
    ('text_svd',vectorizer_svd, 'text'),
    ('text',vectorizer, 'text'),
    ('text_char',vectorizer2, 'text'),
], remainder='drop', n_jobs=-1)



In [ ]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

In [ ]:
svm_clf = SGDClassifier(
    loss='log_loss', 
    penalty='l2',
    alpha=1e-5,
    max_iter=2000,
    tol=1e-6,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
clf = LogisticRegression(
    C=1,                       
    solver='saga',        
    penalty='elasticnet',           
    max_iter=1500,              
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    multi_class='multinomial',
    l1_ratio=0.5
)

voting_clf = VotingClassifier(
    estimators=[ 
        ('svm', svm_clf), 
        ('lr', clf)
    ], n_jobs=-1,
    voting='soft'
)

svm_clf.fit(X_train_final, y_train)
y_pred = svm_clf.predict(X_test_final)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

print(f1_score(y_test, y_pred, average='macro'))



In [ ]:
print(f1_score(y_test, y_pred, average='macro'))

In [ ]:
#submission = pd.DataFrame({
#    'Id': test_ids,    # Usiamo gli ID salvati all'inizio
#    'Predicted': y_pred # O 'label' o 'Category' a seconda delle regole della gara
#})
#
#submission.to_csv('submission_html_pp.csv', index=False)
#print("Fatto! File pronto.")